## Setup & Environment Detection

In [ ]:
import sys

IS_COLAB = 'google.colab' in sys.modules
print(f"Running in Google Colab: {IS_COLAB}")

In [ ]:
import os
import sys

if IS_COLAB:
    print("Running in Google Colab environment.")
    if os.path.exists('/content/aai521_3proj'):
        print("Repository already exists. Pulling latest changes...")
        %cd /content/aai521_3proj
        !git pull
    else:
        print("Cloning repository...")
        !git clone https://github.com/swapnilprakashpatil/aai521_3proj.git
        %cd aai521_3proj    
    %pip install -r requirements.txt --quiet
    sys.path.append('/content/aai521_3proj/src')
else:
    print("Running in local environment.")
    sys.path.append('../src')

## 1. Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Import custom modules
import config
from dataset import FloodDataset, create_dataloaders
from models import create_model
from metrics import SegmentationMetrics
from gpu_manager import GPUManager

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Initialize GPU
gpu_mgr = GPUManager()
gpu_mgr.setup()
device = gpu_mgr.get_device()

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")
if gpu_mgr.is_available():
    print(f"GPU: {gpu_mgr.gpu_name}")
    print(f"GPU Memory: {gpu_mgr.total_memory_gb:.2f} GB")

## 2. Model Selection & Loading

We'll select the best model based on validation performance from the training results.

In [ ]:
# Available models and their expected performance
# Based on IMPROVED_MODELS_SUMMARY.md:
# - FC-Siam-Diff: Expected 45-55% IoU (best for change detection)
# - Siamese U-Net++: Expected 40-50% IoU
# - DeepLabV3+: Expected 40-50% IoU
# - STANet: Expected 40-50% IoU

AVAILABLE_MODELS = {
    'fc_siam_diff': 'FC-Siam-Diff (Change Detection)',
    'siamese_unet++': 'Siamese U-Net++',
    'deeplabv3+': 'DeepLabV3+',
    'unet++': 'U-Net++',
    'segformer': 'SegFormer',
    'stanet': 'STANet'
}

# Select model (change this to use a different model)
SELECTED_MODEL = 'deeplabv3+'  # Best model from training

print(f"Selected Model: {AVAILABLE_MODELS[SELECTED_MODEL]}")
print("\nNote: If you want to use a different model, change SELECTED_MODEL above.")

In [ ]:
# Find the best checkpoint for the selected model
# We'll look for checkpoints in outputs/training/

training_dir = Path('../outputs/training')

# List all training runs for this model
model_dirs = []
if training_dir.exists():
    for d in training_dir.iterdir():
        if d.is_dir() and SELECTED_MODEL in d.name.lower():
            model_dirs.append(d)

if not model_dirs:
    print(f"ERROR: No training checkpoints found for {SELECTED_MODEL}")
    print("Please train the model first using notebook 03_model_training.ipynb")
else:
    # Use the most recent training run
    latest_dir = sorted(model_dirs, key=lambda x: x.name)[-1]
    checkpoint_path = latest_dir / 'checkpoints' / 'best_model.pth'
    
    if checkpoint_path.exists():
        print(f"SUCCESS: Found checkpoint: {checkpoint_path}")
        
        # Load training history to see performance
        history_path = latest_dir / 'training_history.json'
        if history_path.exists():
            with open(history_path, 'r') as f:
                history = json.load(f)
            
            best_epoch = np.argmax(history['val_iou'])
            print(f"\nTraining Summary:")
            print(f"  Best Epoch: {best_epoch + 1}/{len(history['val_iou'])}")
            print(f"  Best Val IoU: {history['val_iou'][best_epoch]:.4f}")
            print(f"  Best Val Dice: {history['val_dice'][best_epoch]:.4f}")
            print(f"  Best Val F1: {history['val_f1'][best_epoch]:.4f}")
    else:
        print(f"ERROR: No best_model.pth found in {latest_dir / 'checkpoints'}")
        checkpoint_path = None

In [ ]:
# Load the model
if checkpoint_path and checkpoint_path.exists():
    print(f"Loading model: {SELECTED_MODEL}...")
    
    # Create model architecture
    model = create_model(
        model_name=SELECTED_MODEL,
        in_channels=6 if 'siamese' not in SELECTED_MODEL.lower() else 3,
        num_classes=config.NUM_CLASSES,
        **config.MODEL_CONFIGS.get(SELECTED_MODEL, {})
    )
    
    # Load checkpoint
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    model.eval()
    
    print(f"SUCCESS: Model loaded successfully")
    print(f"   Parameters: {sum(p.numel() for p in model.parameters()):,}")
else:
    print("WARNING: No checkpoint available. Please train a model first.")
    model = None

## 3. Load Test Data

In [ ]:
# Create test dataloader
print("Loading test dataset...")

_, _, test_loader = create_dataloaders(
    train_dir=config.PROCESSED_TRAIN_DIR,
    val_dir=config.PROCESSED_VAL_DIR,
    test_dir=config.PROCESSED_TEST_DIR,
    batch_size=8,
    num_workers=0,
    pin_memory=False
)

print(f"Test set: {len(test_loader.dataset)} samples ({len(test_loader)} batches)")

## 4. Inference on Test Set

In [ ]:
def run_inference(model, dataloader, device):
    """Run inference on entire dataset and return predictions."""
    model.eval()
    all_predictions = []
    all_targets = []
    all_images = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Running inference"):
            images = batch['image'].to(device)
            masks = batch['mask'].to(device)
            
            # Forward pass
            outputs = model(images)
            
            # Get predictions
            preds = torch.argmax(outputs, dim=1)
            
            # Store results
            all_predictions.append(preds.cpu())
            all_targets.append(masks.cpu())
            all_images.append(images.cpu())
    
    # Concatenate all batches
    all_predictions = torch.cat(all_predictions, dim=0)
    all_targets = torch.cat(all_targets, dim=0)
    all_images = torch.cat(all_images, dim=0)
    
    return all_images, all_predictions, all_targets

In [ ]:
if model is not None:
    print("Running inference on test set...")
    test_images, test_predictions, test_targets = run_inference(model, test_loader, device)
    
    print(f"\nInference complete:")
    print(f"  Images: {test_images.shape}")
    print(f"  Predictions: {test_predictions.shape}")
    print(f"  Targets: {test_targets.shape}")
    print(f"  Unique predictions: {test_predictions.unique().tolist()}")
    print(f"  Unique targets: {test_targets.unique().tolist()}")
else:
    print("WARNING: Skipping inference - no model loaded")

## 5. Quantitative Evaluation

In [ ]:
def compute_metrics(predictions, targets, num_classes, class_names):
    """Compute comprehensive segmentation metrics."""
    metrics_tracker = SegmentationMetrics(num_classes=num_classes)
    
    # Compute metrics
    metrics = metrics_tracker.compute_metrics(predictions, targets)
    
    # Print overall metrics
    print("="*80)
    print("OVERALL TEST SET PERFORMANCE")
    print("="*80)
    print(f"Mean IoU:  {metrics['mean_iou']:.4f}")
    print(f"Mean Dice: {metrics['mean_dice']:.4f}")
    print(f"Mean F1:   {metrics['mean_f1']:.4f}")
    print(f"Accuracy:  {metrics['accuracy']:.4f}")
    
    # Print per-class metrics
    print("\n" + "="*80)
    print("PER-CLASS PERFORMANCE")
    print("="*80)
    
    class_metrics = []
    for i, class_name in enumerate(class_names):
        iou = metrics['iou_per_class'][i]
        dice = metrics['dice_per_class'][i]
        f1 = metrics['f1_per_class'][i]
        precision = metrics['precision_per_class'][i]
        recall = metrics['recall_per_class'][i]
        
        class_metrics.append({
            'Class': class_name,
            'IoU': f"{iou:.4f}",
            'Dice': f"{dice:.4f}",
            'F1': f"{f1:.4f}",
            'Precision': f"{precision:.4f}",
            'Recall': f"{recall:.4f}"
        })
    
    df = pd.DataFrame(class_metrics)
    print(df.to_string(index=False))
    print("="*80)
    
    return metrics, df

In [ ]:
if model is not None:
    print("Computing evaluation metrics...\n")
    test_metrics, metrics_df = compute_metrics(
        test_predictions,
        test_targets,
        config.NUM_CLASSES,
        config.CLASS_NAMES
    )
else:
    print("WARNING: Skipping evaluation - no predictions available")

## 6. Confusion Matrix Analysis

In [ ]:
def plot_confusion_matrix(predictions, targets, class_names, normalize=True):
    """Plot confusion matrix for segmentation results."""
    from sklearn.metrics import confusion_matrix
    
    # Flatten predictions and targets
    preds_flat = predictions.flatten().numpy()
    targets_flat = targets.flatten().numpy()
    
    # Compute confusion matrix
    cm = confusion_matrix(targets_flat, preds_flat, labels=range(len(class_names)))
    
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        fmt = '.3f'
        title = 'Normalized Confusion Matrix'
    else:
        fmt = 'd'
        title = 'Confusion Matrix'
    
    # Plot
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt=fmt, cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                cbar_kws={'label': 'Proportion' if normalize else 'Count'})
    plt.title(title, fontsize=14, fontweight='bold')
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.tight_layout()
    plt.show()
    
    return cm

In [ ]:
if model is not None:
    print("Generating confusion matrix...\n")
    confusion_mat = plot_confusion_matrix(
        test_predictions,
        test_targets,
        config.CLASS_NAMES,
        normalize=True
    )
else:
    print("WARNING: Skipping confusion matrix - no predictions available")

## 7. Qualitative Visualization

In [ ]:
def visualize_predictions(images, predictions, targets, indices, class_names):
    """Visualize predictions vs ground truth for selected samples."""
    num_samples = len(indices)
    fig, axes = plt.subplots(num_samples, 4, figsize=(20, 5*num_samples))
    
    if num_samples == 1:
        axes = axes.reshape(1, -1)
    
    cmap = plt.cm.get_cmap('tab10', len(class_names))
    
    for i, idx in enumerate(indices):
        # Get images
        pre_img = images[idx, :3].permute(1, 2, 0).numpy()
        pre_img = (pre_img - pre_img.min()) / (pre_img.max() - pre_img.min() + 1e-8)
        
        post_img = images[idx, 3:].permute(1, 2, 0).numpy()
        post_img = (post_img - post_img.min()) / (post_img.max() - post_img.min() + 1e-8)
        
        pred = predictions[idx].numpy()
        target = targets[idx].numpy()
        
        # Plot pre-event
        axes[i, 0].imshow(pre_img)
        axes[i, 0].set_title('Pre-Event Image', fontsize=12, fontweight='bold')
        axes[i, 0].axis('off')
        
        # Plot post-event
        axes[i, 1].imshow(post_img)
        axes[i, 1].set_title('Post-Event Image', fontsize=12, fontweight='bold')
        axes[i, 1].axis('off')
        
        # Plot ground truth
        axes[i, 2].imshow(target, cmap=cmap, vmin=0, vmax=len(class_names)-1)
        axes[i, 2].set_title('Ground Truth', fontsize=12, fontweight='bold')
        axes[i, 2].axis('off')
        
        # Plot prediction
        im = axes[i, 3].imshow(pred, cmap=cmap, vmin=0, vmax=len(class_names)-1)
        axes[i, 3].set_title('Prediction', fontsize=12, fontweight='bold')
        axes[i, 3].axis('off')
        
        # Add colorbar to last row
        if i == num_samples - 1:
            cbar = plt.colorbar(im, ax=axes[i, :], orientation='horizontal',
                              pad=0.05, fraction=0.046)
            cbar.set_ticks(range(len(class_names)))
            cbar.set_ticklabels(class_names, rotation=45, ha='right')
    
    plt.tight_layout()
    plt.show()

In [ ]:
if model is not None:
    # Visualize random samples
    print("Visualizing random predictions...\n")
    num_viz = min(5, len(test_predictions))
    random_indices = np.random.choice(len(test_predictions), num_viz, replace=False)
    
    visualize_predictions(
        test_images,
        test_predictions,
        test_targets,
        random_indices,
        config.CLASS_NAMES
    )
else:
    print("WARNING: Skipping visualization - no predictions available")

## 8. Best and Worst Predictions

Identify samples with highest and lowest IoU to understand model strengths and weaknesses.

In [ ]:
def compute_sample_iou(predictions, targets):
    """Compute IoU for each sample."""
    ious = []
    
    for pred, target in zip(predictions, targets):
        # Compute IoU for this sample
        intersection = (pred == target).sum().item()
        union = pred.numel()
        iou = intersection / union if union > 0 else 0.0
        ious.append(iou)
    
    return np.array(ious)

In [ ]:
if model is not None:
    # Compute per-sample IoU
    sample_ious = compute_sample_iou(test_predictions, test_targets)
    
    # Find best and worst samples
    best_indices = np.argsort(sample_ious)[-3:][::-1]  # Top 3
    worst_indices = np.argsort(sample_ious)[:3]  # Bottom 3
    
    print("\n" + "="*80)
    print("BEST PREDICTIONS (Highest IoU)")
    print("="*80)
    for rank, idx in enumerate(best_indices, 1):
        print(f"{rank}. Sample {idx}: IoU = {sample_ious[idx]:.4f}")
    
    print("\n" + "="*80)
    print("WORST PREDICTIONS (Lowest IoU)")
    print("="*80)
    for rank, idx in enumerate(worst_indices, 1):
        print(f"{rank}. Sample {idx}: IoU = {sample_ious[idx]:.4f}")
    print("="*80)
    
    # Visualize best predictions
    print("\nBest Predictions:")
    visualize_predictions(
        test_images,
        test_predictions,
        test_targets,
        best_indices,
        config.CLASS_NAMES
    )
    
    # Visualize worst predictions
    print("\nWorst Predictions:")
    visualize_predictions(
        test_images,
        test_predictions,
        test_targets,
        worst_indices,
        config.CLASS_NAMES
    )
else:
    print("WARNING: Skipping best/worst analysis - no predictions available")

## 9. Error Analysis

In [ ]:
if model is not None:
    print("\n" + "="*80)
    print("ERROR ANALYSIS")
    print("="*80)
    
    # Compute class-wise error statistics
    preds_flat = test_predictions.flatten().numpy()
    targets_flat = test_targets.flatten().numpy()
    
    print("\nClass Distribution:")
    print("-" * 80)
    for i, class_name in enumerate(config.CLASS_NAMES):
        true_count = (targets_flat == i).sum()
        pred_count = (preds_flat == i).sum()
        true_pct = 100 * true_count / len(targets_flat)
        pred_pct = 100 * pred_count / len(preds_flat)
        
        print(f"{class_name}:")
        print(f"  Ground Truth: {true_count:,} pixels ({true_pct:.2f}%)")
        print(f"  Predicted:    {pred_count:,} pixels ({pred_pct:.2f}%)")
        print()
    
    # Analyze misclassifications
    print("\nCommon Misclassifications:")
    print("-" * 80)
    
    for true_class in range(config.NUM_CLASSES):
        mask = targets_flat == true_class
        if mask.sum() > 0:
            misclassified = preds_flat[mask]
            unique, counts = np.unique(misclassified, return_counts=True)
            
            print(f"\n{config.CLASS_NAMES[true_class]} misclassified as:")
            for pred_class, count in sorted(zip(unique, counts), key=lambda x: -x[1]):
                if pred_class != true_class:
                    pct = 100 * count / mask.sum()
                    if pct > 1.0:  # Only show if >1%
                        print(f"  - {config.CLASS_NAMES[pred_class]}: {count:,} pixels ({pct:.2f}%)")
    
    print("\n" + "="*80)
else:
    print("WARNING: Skipping error analysis - no predictions available")

## 10. Save Results

In [ ]:
if model is not None:
    # Create results directory
    results_dir = Path('../outputs/final_results')
    results_dir.mkdir(parents=True, exist_ok=True)
    
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    results_file = results_dir / f'{SELECTED_MODEL}_results_{timestamp}.json'
    
    # Prepare results dictionary
    results = {
        'model': SELECTED_MODEL,
        'model_name': AVAILABLE_MODELS[SELECTED_MODEL],
        'timestamp': timestamp,
        'checkpoint': str(checkpoint_path),
        'test_set_size': len(test_predictions),
        'overall_metrics': {
            'mean_iou': float(test_metrics['mean_iou']),
            'mean_dice': float(test_metrics['mean_dice']),
            'mean_f1': float(test_metrics['mean_f1']),
            'accuracy': float(test_metrics['accuracy'])
        },
        'per_class_metrics': metrics_df.to_dict('records'),
        'sample_ious': {
            'mean': float(sample_ious.mean()),
            'std': float(sample_ious.std()),
            'min': float(sample_ious.min()),
            'max': float(sample_ious.max()),
            'median': float(np.median(sample_ious))
        }
    }
    
    # Save to JSON
    with open(results_file, 'w') as f:
        json.dump(results, f, indent=2)
    
    print(f"\nSUCCESS: Results saved to: {results_file}")
    
    # Also save metrics DataFrame as CSV
    csv_file = results_dir / f'{SELECTED_MODEL}_metrics_{timestamp}.csv'
    metrics_df.to_csv(csv_file, index=False)
    print(f"SUCCESS: Metrics CSV saved to: {csv_file}")
else:
    print("WARNING: Skipping save - no results to save")

## 11. Final Project Summary Report

In [ ]:
if model is not None:
    print("\n" + "="*80)
    print("FLOOD DETECTION PROJECT - FINAL SUMMARY")
    print("="*80)
    
    print("\nPROJECT OVERVIEW")
    print("-" * 80)
    print("Dataset: SpaceNet8 - Flood Detection Challenge")
    print("Task: Multiclass Semantic Segmentation")
    print(f"Classes: {config.NUM_CLASSES} ({', '.join(config.CLASS_NAMES)})")
    print(f"Training Samples: {len(train_loader.dataset) if 'train_loader' in dir() else 'N/A'}")
    print(f"Validation Samples: {len(val_loader.dataset) if 'val_loader' in dir() else 'N/A'}")
    print(f"Test Samples: {len(test_loader.dataset)}")
    
    print("\nMODEL INFORMATION")
    print("-" * 80)
    print(f"Architecture: {AVAILABLE_MODELS[SELECTED_MODEL]}")
    print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Checkpoint: {checkpoint_path.name}")
    
    print("\nPERFORMANCE METRICS")
    print("-" * 80)
    print(f"Test Set Mean IoU:  {test_metrics['mean_iou']:.4f}")
    print(f"Test Set Mean Dice: {test_metrics['mean_dice']:.4f}")
    print(f"Test Set Mean F1:   {test_metrics['mean_f1']:.4f}")
    print(f"Test Set Accuracy:  {test_metrics['accuracy']:.4f}")
    
    print("\nPER-CLASS PERFORMANCE")
    print("-" * 80)
    for i, class_name in enumerate(config.CLASS_NAMES):
        iou = test_metrics['iou_per_class'][i]
        print(f"{class_name:20s}: IoU = {iou:.4f}")
    
    print("\nSAMPLE STATISTICS")
    print("-" * 80)
    print(f"Mean IoU per sample:   {sample_ious.mean():.4f}")
    print(f"Std IoU per sample:    {sample_ious.std():.4f}")
    print(f"Min IoU per sample:    {sample_ious.min():.4f}")
    print(f"Max IoU per sample:    {sample_ious.max():.4f}")
    print(f"Median IoU per sample: {np.median(sample_ious):.4f}")
    
    print("\nKEY FINDINGS")
    print("-" * 80)
    
    # Find best performing class
    best_class_idx = np.argmax(test_metrics['iou_per_class'])
    worst_class_idx = np.argmin(test_metrics['iou_per_class'])
    
    print(f"1. Best performing class: {config.CLASS_NAMES[best_class_idx]} "
          f"(IoU: {test_metrics['iou_per_class'][best_class_idx]:.4f})")
    print(f"2. Most challenging class: {config.CLASS_NAMES[worst_class_idx]} "
          f"(IoU: {test_metrics['iou_per_class'][worst_class_idx]:.4f})")
    print(f"3. Sample consistency: {sample_ious.std():.4f} std deviation")
    
    print("\nRECOMMENDATIONS")
    print("-" * 80)
    print("1. For production deployment:")
    print("   - Use ensemble of top 3 models for better robustness")
    print("   - Implement post-processing (CRF, morphological operations)")
    print("   - Add test-time augmentation for edge cases")
    
    print("\n2. For further improvement:")
    print("   - Collect more training data for minority classes")
    print("   - Experiment with multi-scale inference")
    print("   - Try self-supervised pretraining on flood imagery")
    
    print("\n3. For real-world deployment:")
    print("   - Optimize model for inference speed (quantization, pruning)")
    print("   - Build confidence estimation for predictions")
    print("   - Create user-friendly visualization tools")
    
    print("\n" + "="*80)
    print("PROJECT COMPLETE!")
    print("="*80)
else:
    print("\nWARNING: Cannot generate final summary - no model evaluated")
    print("Please train a model first using notebook 03_model_training.ipynb")

## 12. Export Results for Submission

Generate prediction files in the format required for SpaceNet8 submission.

In [ ]:
if model is not None:
    # Create submission directory
    submission_dir = Path('../outputs/submission')
    submission_dir.mkdir(parents=True, exist_ok=True)
    
    # Save predictions as numpy arrays
    predictions_file = submission_dir / f'{SELECTED_MODEL}_predictions.npy'
    np.save(predictions_file, test_predictions.numpy())
    
    print(f"SUCCESS: Predictions saved to: {predictions_file}")
    print(f"   Shape: {test_predictions.shape}")
    print(f"   File size: {predictions_file.stat().st_size / 1e6:.2f} MB")
    
    # Create submission metadata
    metadata = {
        'model': SELECTED_MODEL,
        'num_samples': len(test_predictions),
        'num_classes': config.NUM_CLASSES,
        'class_names': config.CLASS_NAMES,
        'mean_iou': float(test_metrics['mean_iou']),
        'timestamp': timestamp
    }
    
    metadata_file = submission_dir / f'{SELECTED_MODEL}_metadata.json'
    with open(metadata_file, 'w') as f:
        json.dump(metadata, f, indent=2)
    
    print(f"SUCCESS: Metadata saved to: {metadata_file}")
else:
    print("WARNING: Skipping export - no predictions to export")

## 13. Next Steps & Future Work

### Immediate Actions:
1. **Model Ensemble**: Combine predictions from top 3 models
2. **Post-Processing**: Apply CRF or morphological operations
3. **Test-Time Augmentation**: Flip/rotate images during inference

### Future Improvements:
1. **Data Augmentation**: More aggressive augmentation for minority classes
2. **Semi-Supervised Learning**: Use unlabeled flood images
3. **Active Learning**: Identify and label hard examples
4. **Multi-Task Learning**: Predict flood depth/severity simultaneously

### Deployment Considerations:
1. **Model Optimization**: Quantization, pruning, ONNX export
2. **API Development**: REST API for inference service
3. **Monitoring**: Track prediction quality in production
4. **Retraining Pipeline**: Automated retraining with new data

---

**Thank you for using this flood detection system!**